In [ ]:
# Google Colab requires Drive mounting before accessing the NIH metadata files.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install the packages required by this notebook and the shared preprocessing utilities.
%pip install -q numpy pandas scikit-learn matplotlib jupyter


## Configuration — Google Drive Paths

The next code cell defines the only machine-specific setting, `BASE_DRIVE_PATH`, and validates the three NIH metadata files. This notebook does not require an image directory and does not download or copy X-ray images.

# NIH ChestX-ray14 preprocessing

This notebook creates the NIH federated client as metadata-only CSV files. It keeps clean target positives and explicit `No Finding` negatives, excludes ambiguous label combinations, preserves patient IDs, and splits whole patients. Images are never copied or downloaded in this phase.

**Storage model:** Code and notebooks are stored in GitHub, while the large NIH metadata files and future image files are stored in Google Drive. The actual X-ray images will be downloaded later, after the target disease and required sample size are finalized.

In [ ]:
from pathlib import Path
import json, sys, platform
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir() and Path('/content').exists():
    PROJECT_ROOT = Path('/content') / 'adaptive-federated-healthcare'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from dataset_utils import nih_clean_binary_labels, standardize_nih
from preprocessing_utils import (assert_no_leakage, assert_no_study_leakage, build_summary,
                                  require_file, sample_clean_cases, save_splits,
                                  save_summary, set_reproducibility, split_by_patient)

set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, 'numpy:', np.__version__)

# Configuration - Google Drive Paths
BASE_DRIVE_PATH = "/content/drive/MyDrive/Healthcare_FL"
NIH_ROOT = f"{BASE_DRIVE_PATH}/raw/NIH"
NIH_METADATA_DIR = f"{NIH_ROOT}/metadata"

METADATA_FILE = f"{NIH_METADATA_DIR}/Data_Entry_2017.csv"
TRAIN_VAL_FILE = f"{NIH_METADATA_DIR}/train_val_list.txt"
TEST_FILE = f"{NIH_METADATA_DIR}/test_list.txt"

# Outputs are metadata only and are kept in Drive rather than committed to GitHub.
OUTPUT_DIR = Path(BASE_DRIVE_PATH) / 'processed' / 'client_01_NIH'
metadata_path = require_file(Path(METADATA_FILE), 'NIH metadata CSV')
train_val_path = require_file(Path(TRAIN_VAL_FILE), 'NIH train/validation split list')
test_path = require_file(Path(TEST_FILE), 'NIH test split list')

print('NIH metadata path:', METADATA_FILE)
print('NIH train/validation split path:', TRAIN_VAL_FILE)
print('NIH test split path:', TEST_FILE)
print('All required NIH metadata files exist.')
print('Target disease:', TARGET_DISEASE)

## Load and inspect
NIH `Finding Labels` is pipe-separated. The clean binary policy is deliberately conservative: a target-containing study is positive; exactly `No Finding` is negative; every other combination is excluded.

In [ ]:
nih = pd.read_csv(metadata_path)
train_val_ids = pd.read_csv(train_val_path, header=None, names=['image_id'])
test_ids = pd.read_csv(test_path, header=None, names=['image_id'])

print('images:', len(nih))
print('unique patients:', nih['Patient ID'].nunique())
print('columns:', nih.columns.tolist())
print('official train/validation image IDs:', len(train_val_ids))
print('official test image IDs:', len(test_ids))
print('official split image overlap:', len(set(train_val_ids['image_id']) & set(test_ids['image_id'])))
display(nih.isna().sum().sort_values(ascending=False).head(20).to_frame('missing'))
display(nih['Finding Labels'].value_counts(dropna=False).head(20).to_frame('label_count'))
clean, counts = nih_clean_binary_labels(nih, TARGET_DISEASE)
print('Counts before filtering:', counts)
print('Clean rows:', len(clean))

In [ ]:
# No image directory is required for this metadata-only phase.
standardized = standardize_nih(clean, image_dir=None)
standardized = sample_clean_cases(standardized, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES, RANDOM_SEED)
print('Rows after optional sampling:', len(standardized))
display(standardized.head())
print(standardized['target'].value_counts().sort_index())
splits = split_by_patient(standardized, seed=RANDOM_SEED)
assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient leakage checks: passed')
print('study leakage checks: passed where study_id exists')
save_splits(splits, OUTPUT_DIR)
summary = build_summary(splits, 'NIH', {'target_disease': TARGET_DISEASE, 'random_seed': RANDOM_SEED, 'max_positive_samples': MAX_POSITIVE_SAMPLES, 'max_negative_samples': MAX_NEGATIVE_SAMPLES, 'filter_counts_before_filtering': counts, 'official_split_lists_loaded': True, 'official_split_used': False, 'image_paths_materialized': False})
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print('Saved:', sorted(path.name for path in OUTPUT_DIR.glob('*')))